In [7]:
from pathlib import Path
import hashlib
import soundfile as sf
import numpy as np
import pandas as pd

OLD_ROOT = Path("datasets_compare/old")
NEW_ROOT = Path("datasets_compare/new")

DATASETS = [
    "prepared_words",
    "unknown_balanced",
    "test_ready_chjem1",
    "test_ready_chjem2",
]

def md5_file(path: Path) -> str:
    h = hashlib.md5()
    h.update(path.read_bytes())
    return h.hexdigest()

def audio_info(path: Path):
    x, fs = sf.read(str(path), always_2d=False)
    x = np.asarray(x)

    if x.ndim > 1:
        x = x.mean(axis=1)

    return {
        "fs": fs,
        "n": len(x),
        "duration_s": len(x) / fs,
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "peak": float(np.max(np.abs(x))),
        "rms": float(np.sqrt(np.mean(x.astype(np.float64) ** 2) + 1e-12)),
        "md5": md5_file(path),
    }

def build_manifest(root: Path):
    rows = []

    for wav in sorted(root.rglob("*.wav")):
        rel = wav.relative_to(root).as_posix()
        label = wav.parent.name

        info = audio_info(wav)
        info["rel"] = rel
        info["label"] = label
        rows.append(info)

    return pd.DataFrame(rows)

for ds in DATASETS:
    print("\n" + "=" * 80)
    print(ds)

    old_dir = OLD_ROOT / ds
    new_dir = NEW_ROOT / ds

    old = build_manifest(old_dir)
    new = build_manifest(new_dir)

    old_rels = set(old["rel"])
    new_rels = set(new["rel"])

    only_old = sorted(old_rels - new_rels)
    only_new = sorted(new_rels - old_rels)
    common = sorted(old_rels & new_rels)

    print("old files:", len(old))
    print("new files:", len(new))
    print("only old:", len(only_old))
    print("only new:", len(only_new))

    old_map = old.set_index("rel").to_dict("index")
    new_map = new.set_index("rel").to_dict("index")

    same_name_diff_md5 = []
    same_name_diff_len = []
    same_name_diff_rms = []

    for rel in common:
        if old_map[rel]["md5"] != new_map[rel]["md5"]:
            same_name_diff_md5.append(rel)

        if old_map[rel]["n"] != new_map[rel]["n"]:
            same_name_diff_len.append(rel)

        if abs(old_map[rel]["rms"] - new_map[rel]["rms"]) > 1e-6:
            same_name_diff_rms.append(rel)

    print("same filename but different audio:", len(same_name_diff_md5))
    print("same filename but different length:", len(same_name_diff_len))
    print("same filename but different RMS:", len(same_name_diff_rms))

    if only_old:
        print("\nExamples only old:")
        for x in only_old[:10]:
            print(" ", x)

    if only_new:
        print("\nExamples only new:")
        for x in only_new[:10]:
            print(" ", x)

    if same_name_diff_md5:
        print("\nExamples same filename but different content:")
        for x in same_name_diff_md5[:10]:
            print(" ", x)

    # Save manifests
    old.to_csv(f"manifest_old_{ds}.csv", index=False)
    new.to_csv(f"manifest_new_{ds}.csv", index=False)


prepared_words
old files: 300
new files: 300
only old: 0
only new: 0
same filename but different audio: 0
same filename but different length: 0
same filename but different RMS: 0

unknown_balanced
old files: 300
new files: 300
only old: 0
only new: 0
same filename but different audio: 0
same filename but different length: 0
same filename but different RMS: 0

test_ready_chjem1
old files: 140
new files: 140
only old: 0
only new: 0
same filename but different audio: 0
same filename but different length: 0
same filename but different RMS: 0

test_ready_chjem2
old files: 141
new files: 140
only old: 1
only new: 0
same filename but different audio: 0
same filename but different length: 0
same filename but different RMS: 0

Examples only old:
  hurtig/test_får_chjemfrfr.001.wav
